# ISA2 – Property price model (Venta)
Notebook version of `read.py` (data cleaning) and `transform.py` (features + Ridge model), following the SageMaker workflow: local run → training script → SageMaker estimator → hyperparameter tuning → endpoint.

Upload your `Data/` folder next to this notebook before running (same relative paths as the scripts: `./Data/IIEG/...`, `./Data/Clean/...`, `./Data/Process/helpers/...`, `./Data/CORREOS/...`).

Install the libraries the cleaning steps use that are not in the default SageMaker image.

In [0]:
import os
import glob
import re
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import numpy as np
import jmespath
import geopandas as gpd
from math import radians, cos, sin, asin, sqrt

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker import get_execution_role
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter
from sagemaker.session import Session
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

sagemaker_session = Session()
role = get_execution_role()

bucket_name = "sagemaker-us-east-2-205952664736"
bucket_uri = "s3://sagemaker-us-east-2-205952664736"
# Initialize S3 client
s3_client = boto3.client("s3", region_name=boto3.Session().region_name)
boto_sess = boto3.Session(region_name="us-east-2")


sagemaker_session = sagemaker.Session(boto_session=boto_sess, default_bucket=bucket_name,default_bucket_prefix='')

def upload_to_s3(local_path, key):
    # Open in binary mode ('rb') so boto3 put_object receives bytes, not str
    with open(local_path, "rb") as f:
        s3_client.put_object(Body=f, Bucket=bucket_name, Key=key)
    return f"s3://{bucket_name}/{key}"




sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket


sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


## Part 1 – Data cleaning (`read.py`)
Each cell defines one step of the pipeline. The cell **Run the cleaning pipeline** calls them in order; comment out the steps you have already run.

Read every IIEG Excel file (2020–2024), normalize headers and text, add Quarter, Year and price per m² (`ppm2`), drop duplicates and write `ALL_Venta.csv`.

In [0]:

def create_frames_year_all():
    
    response = s3_client.list_objects(Bucket=bucket_name)
    files = jmespath.search("Contents[].Key[]" , response)

    print(files)

    sheets = ["Venta"]
    all_frames = []
    for i in range(2020, 2025):
        frames = []
        for file in files:
            if str(i) in file:
                print(file)

                match = re.search(r"_(\d+)_(\d{4}).xlsx$", file)

                if match:
                    month, year = match.groups()
                    month = month.lstrip("0")

                for s in sheets:
                    print(s)
                    raw = pd.read_excel(bucket_uri+'/'+file, s)

                    if i in range(2020, 2022) or (
                        int(year) == 2022 and int(month) in range(0, 8)
                    ):
                        header_row = raw.index[raw.eq("Inmueble").any(axis=1)][0]
                        df = raw.iloc[header_row + 1 :].copy()
                        df.columns = raw.iloc[header_row]
                        df = df.reset_index(drop=True).dropna(how="all")

                    else:
                        df = raw

                    print(df.head())

                    df.rename(
                        columns={
                            "Inmueble": "Type",
                            "Precio": "Price",
                            "Municipio": "Municipality",
                            "Colonia": "Colony",
                            "Metros cuadrados de terreno": "Terrain",
                            "Metros cuadrados de construcción": "Construction",
                        },
                        inplace=True,
                    )

                    df.drop("Construction", axis=1, inplace=True, errors="Ignore")
                    df["Quarter"] = (int(month) + 2) // 3
                    df["Year"] = year
                    df["ppm2"] = df["Price"] / df["Terrain"]

                    text_cols = df.select_dtypes(include="string").columns

                    for column in text_cols:
                        df[column] = (
                            df[column]
                            .str.normalize("NFKD")
                            .str.encode("ascii", errors="ignore")
                            .str.decode("utf-8")
                            .str.strip()
                            .str.upper()
                        )

                    df.reset_index(inplace=True, drop=True)
                    df.dropna(inplace=True)

                    print(df.head())
                    frames.append(df)

        year = pd.concat(frames)

        all_frames.append(year)

    all_df = pd.concat(all_frames)

    all_df = all_df.loc[~(all_df.drop(columns=["Year", "Quarter"])).duplicated()]

    mask = (
        all_df["Terrain"].lt(20)
        & all_df["Terrain"].gt(1250)
        & all_df["ppm2"].gt(100000)
        & all_df["ppm2"].lt(2500)
    )

    all_df = all_df[~mask]
    all_df.dropna(inplace=True)
    all_df.to_csv("./Data/Clean/ALL_Venta.csv", index=False)

create_frames_year_all()

['Data/', 'Data/Clean/', 'Data/Process/', 'RAW/', 'RAW/BD_SECTOR_INMOBILIARIO_01_2023.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_01_2024.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_04_2023.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_04_2024.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_07_2023.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_07_2024(1).xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_10_2020.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_10_2021.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_10_2022.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_10_2023.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_1_2021.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_1_2022.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_4_2021.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_4_2022.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_7_2020.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_7_2021.xlsx', 'RAW/BD_SECTOR_INMOBILIARIO_7_2022.xlsx', 'RAW/shf.xlsx']
RAW/BD_SECTOR_INMOBILIARIO_10_2020.xlsx
Venta


4 Inmueble  ... Metros cuadrados de construcción
0     Casa  ...                              NaN
1     Casa  ...                              NaN
2     Casa  ...                              NaN
3     Casa  ...                              NaN
4     Casa  ...                              NaN

[5 rows x 6 columns]
4   Type Municipality    Price  ... Quarter  Year          ppm2
6   Casa  GUADALAJARA   470234  ...       4  2020   6269.786667
12  Casa       TONALÁ   567500  ...       4  2020   9458.333333
13  Casa       TONALÁ   567500  ...       4  2020   9458.333333
23  Casa  TLAQUEPAQUE   830000  ...       4  2020  12205.882353
55  Casa  TLAQUEPAQUE  1600000  ...       4  2020  12307.692308

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_7_2020.xlsx
Venta


4      Inmueble  ... Metros cuadrados de construcción
0          Casa  ...                             5233
1          Casa  ...                             5266
2          Casa  ...                              NaN
3          Casa  ...                              NaN
4  Departamento  ...                             4815

[5 rows x 6 columns]
4            Type Municipality    Price  ... Quarter  Year         ppm2
21           Casa  TLAQUEPAQUE   840000  ...       3  2020      10500.0
150  Departamento  GUADALAJARA  5688320  ...       3  2020      28160.0
154  Departamento      ZAPOPAN  1750000  ...       3  2020    530.30303
163          Casa      ZAPOPAN   309000  ...       3  2020    78.806427
166          Casa     EL SALTO   573485  ...       3  2020  1120.087891

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_10_2021.xlsx
Venta


4      Inmueble  ... Metros cuadrados de construcción
0          Casa  ...                              400
1          Casa  ...                              235
2  Departamento  ...                              NaN
3  Departamento  ...                              259
4          Casa  ...                              210

[5 rows x 6 columns]
4          Type Municipality     Price  ... Quarter  Year          ppm2
3  Departamento      ZAPOPAN  14287350  ...       4  2021  55163.513514
4          Casa      ZAPOPAN   4800000  ...       4  2021  33333.333333
7  Departamento      ZAPOPAN   5060000  ...       4  2021       55000.0
8          Casa   TLAJOMULCO   1300000  ...       4  2021  14444.444444
9          Casa      ZAPOPAN    950000  ...       4  2021        5937.5

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_1_2021.xlsx
Venta


4      Inmueble  ... Metros cuadrados de construcción
0          Casa  ...                              NaN
1          Casa  ...                              NaN
2  Departamento  ...                              NaN
3          Casa  ...                              NaN
4          Casa  ...                              NaN

[5 rows x 6 columns]
4  Type Municipality       Price  ... Quarter  Year           ppm2
1  Casa      ZAPOPAN  14587896.6  ...       1  2021    14443.46198
3  Casa   TLAJOMULCO    19967420  ...       1  2021  221860.222222
4  Casa   TLAJOMULCO    19967420  ...       1  2021   13885.549374
5  Casa   TLAJOMULCO    19967420  ...       1  2021   24681.606922
6  Casa   TLAJOMULCO    11980452  ...       1  2021   16389.127223

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_4_2021.xlsx
Venta


4 Inmueble  ... Metros cuadrados de construcción
0     Casa  ...                             1292
1     Casa  ...                             1950
2     Casa  ...                              NaN
3     Casa  ...                             1600
4     Casa  ...                             1600

[5 rows x 6 columns]
4  Type Municipality     Price  ... Quarter  Year          ppm2
0  Casa      ZAPOPAN  95000000  ...       2  2021  58787.128713
1  Casa      ZAPOPAN  90000000  ...       2  2021  11553.273427
2  Casa      ZAPOPAN  90000000  ...       2  2021       12000.0
3  Casa  GUADALAJARA  89000000  ...       2  2021  39138.082674
4  Casa  GUADALAJARA  89000000  ...       2  2021  39138.082674

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_7_2021.xlsx
Venta


... (8 outputs left out)

       Inmueble  ... Metros cuadrados de construcción
0  Departamento  ...                              NaN
1          Casa  ...                       114.980003
2  Departamento  ...                              NaN
3          Casa  ...                        68.000000
4  Departamento  ...                              NaN

[5 rows x 6 columns]
    Type Municipality       Price  ... Quarter  Year          ppm2
1   Casa   TLAJOMULCO   1550000.0  ...       4  2023  15656.565657
3   Casa      ZAPOPAN   1990000.0  ...       4  2023  28028.169014
8   Casa      ZAPOPAN   1210000.0  ...       4  2023   9680.000000
10  Casa   TLAJOMULCO  14500000.0  ...       4  2023  27566.539924
17  Casa      ZAPOPAN   3950000.0  ...       4  2023  41145.833333

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_01_2024.xlsx
Venta


       Inmueble  ... Metros cuadrados de construcción
0          Casa  ...                            110.0
1          Casa  ...                            130.0
2          Casa  ...                            175.0
3          Casa  ...                            605.0
4  Departamento  ...                             60.0

[5 rows x 6 columns]
    Type Municipality      Price  ... Quarter  Year          ppm2
1   Casa      ZAPOPAN  2480000.0  ...       1  2024  23619.047619
2   Casa   TLAJOMULCO  4500000.0  ...       1  2024  32142.857143
6   Casa      ZAPOPAN  2987580.0  ...       1  2024  36433.902439
9   Casa  GUADALAJARA  7850000.0  ...       1  2024  41978.609626
11  Casa      ZAPOPAN  5390000.0  ...       1  2024  51333.333333

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_04_2024.xlsx
Venta


  Inmueble  ...  Metros cuadrados de construcción
0     Casa  ...                               NaN
1     Casa  ...                               NaN
2     Casa  ...                               NaN
3     Casa  ...                               NaN
4     Casa  ...                               NaN

[5 rows x 6 columns]
            Type    Price Municipality  ... Quarter  Year          ppm2
1           Casa   910000      ZAPOPAN  ...       2  2024   4136.363636
7   Departamento  3900000      ZAPOPAN  ...       2  2024  36448.598131
9           Casa  9190000  GUADALAJARA  ...       2  2024  36760.000000
10  Departamento  2350000  GUADALAJARA  ...       2  2024  39166.666667
11  Departamento  3453683  GUADALAJARA  ...       2  2024  46049.106667

[5 rows x 8 columns]
RAW/BD_SECTOR_INMOBILIARIO_07_2024(1).xlsx
Venta


       Inmueble  ... Metros cuadrados de construcción
0  Departamento  ...                            190.0
1  Departamento  ...                             90.0
2          Casa  ...                            450.0
3          Casa  ...                            460.0
4  Departamento  ...                             87.0

[5 rows x 6 columns]
           Type Municipality  ...  Year          ppm2
3          Casa      ZAPOPAN  ...  2024  31418.918919
5          Casa  GUADALAJARA  ...  2024  24583.333333
6  Departamento  GUADALAJARA  ...  2024  54444.444444
8          Casa  GUADALAJARA  ...  2024  35000.000000
9  Departamento  GUADALAJARA  ...  2024  50515.009174

[5 rows x 8 columns]


Add the SHF housing price index (Jalisco) by Year, Quarter and Municipality.

In [0]:
def add_shf_index():
    
    file = bucket_uri+'/'+"RAW/shf.csv"

    df = pd.read_csv(file)

    df = df.loc[df["Estado"] == "Jalisco"]

    df["Municipio"] = (
        df["Municipio"]
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.strip()
        .str.upper()
    )

    df.rename(
        columns={
            "Año": "Year",
            "Municipio": "Municipality",
            "Trimestre": "Quarter",
            "Indice": "shf_index",
        },
        inplace=True,
    )

    df.loc[df["Municipality"] == "TLAJOMULCO DE ZUNIGA", "Municipality"] = "TLAJOMULCO"
    df.loc[df["Municipality"] == "SAN PEDRO TLAQUEPAQUE", "Municipality"] = "TLAQUEPAQUE"

    df.dropna(subset=["Municipality"], inplace=True)

    venta_df = pd.read_csv("./Data/Clean/ALL_Venta.csv")
    venta_df.drop(columns=["shf_index"], inplace=True, errors="ignore")

    venta_df = venta_df.merge(
        df[["Year", "Municipality", "Quarter", "shf_index"]],
        on=["Quarter", "Year", "Municipality"],
        how="left",
    )

    venta_df.dropna(inplace=True)
    #venta_df.to_csv("./Data/Clean/ALL_Venta.csv", index=False)

    df.to_csv("./shf.csv",index=False)
    upload_to_s3("./shf.csv", "Data/Clean/shf.csv")

add_shf_index()

In [0]:
res = upload_to_s3("./shf.csv", "Data/Clean/shf.csv")

Build the unique (Municipality, Colony) list and find likely typos with Levenshtein distance.

In [0]:
def make_cp_dict_unclean():
    path = "./Data/Clean/"
    files = glob.glob(path + "ALL_*.csv")

    frames = list()

    for file in files:
        print(file)
        df = pd.read_csv(file)

        uniques = df.drop_duplicates(subset=["Municipality", "Colony"])[
            ["Municipality", "Colony"]
        ]

        print(uniques.head())

        frames.append(uniques)

    cp_df = pd.concat(frames)
    cp_df.drop_duplicates(inplace=True)
    cp_df.to_csv("./Data/Process/helpers/cp_dict_unclean.csv", index=False)
make_cp_dict_unclean()

def check_typos_cp_dict():

    import Levenshtein

    cp = pd.read_csv("./Data/Process/helpers/cp_dict_unclean.csv")
    # Create an empty list to store the match scores
    scores = []

    for index1, row1 in cp.iterrows():
        for index2, row2 in cp.iterrows():
            # Calculate the match score using Levenshtein.distance
            if index1 > index2 or index1 == index2:
                continue

            score = Levenshtein.distance(row1["Colony"], row2["Colony"])
            if 0 < score < 4 and row1["Municipality"] == row2["Municipality"]:
                scores.append({"index1": index1, "index2": index2, "score": score})

    # Create a new DataFrame from the scores list
    scores_df = pd.DataFrame(scores)

    scores_df.to_csv("./Data/Process/helpers/cp_helper.csv")

./Data/Clean/ALL_Venta.csv
  Municipality                         Colony
0  GUADALAJARA             PARQUES DEL NILO I
1  TLAQUEPAQUE                    SANTA ANITA
2  TLAQUEPAQUE  PARQUES DE STA CRUZ DEL VALLE
4  GUADALAJARA              JARDINES DEL SAUZ
5   TLAJOMULCO            FORESTA SANTA ANITA


Apply `source_replacement_map.csv` (colony typo fixes + CP) to the colony dictionary, and replace each listing's colony with its CP in `ALL_Venta.csv`.

In [0]:
def clean_cp_dict():
    # Replacement map built with Claude from cp_helper.csv, sepomex.csv and
    # ISDCOL2020_2024.xlsx (typos, numbered colonies, nearest CP match).
    cp_dict = pd.read_csv("./Data/Process/helpers/cp_dict_unclean.csv")

    replace_df = pd.read_csv(bucket_uri+"/Data/Process/source_replacement_map.csv")
    replace_df.rename({"new_cp": "CP"}, inplace=True, axis=1)

    cp_dict = cp_dict.merge(
        replace_df[["Municipality", "original_colony", "new_colony", "CP"]],
        left_on=["Municipality", "Colony"],
        right_on=["Municipality", "original_colony"],
        how="left",
    )

    cp_dict["Colony"] = cp_dict["new_colony"]
    cp_dict.drop(["original_colony", "new_colony"], axis=1, inplace=True)
    cp_dict.drop_duplicates(subset=["Colony", "Municipality", "CP"], inplace=True)
    cp_dict.dropna(subset=["CP"], inplace=True)
    cp_dict["CP"] = cp_dict["CP"].astype("int32")
    cp_dict.to_csv("./Data/Clean/cp_dict.csv", index=False)

#clean_cp_dict()

def clean_ALL_colonies():

    replace_df = pd.read_csv(bucket_uri+"/Data/Process/source_replacement_map.csv")
    replace_df.rename({"new_cp": "CP"}, inplace=True, axis=1)
    file = "./Data/Clean/ALL_Venta.csv"

    df = pd.read_csv(file)
    print(df.keys())
    df = df.merge(
        replace_df[["Municipality", "original_colony", "CP"]],
        left_on=["Municipality", "Colony"],
        right_on=["Municipality", "original_colony"],
        how="left",
    )

    df.drop(["original_colony", "Colony"], axis=1, inplace=True)
    df.dropna(inplace=True)

    df = df.loc[~(df.drop(columns=["Year", "Quarter"])).duplicated()]
    df.to_csv(file, index=False)
    print(df.head)

#clean_ALL_colonies()

Index(['Type', 'Municipality', 'Price', 'Terrain', 'Quarter', 'Year', 'ppm2',
       'shf_index', 'CP'],
      dtype='object')


KeyError: 'Colony'

Coordinates for each CP from `./Data/Clean/14-Jal.geojson` (centroid, or an inside point when the centroid falls outside the shape), then add them to `ALL_Venta.csv`.

In [0]:
cp_dict = pd.read_csv("./Data/Clean/cp_dict.csv")
print(cp_dict.head)

<bound method NDFrame.head of      Municipality                           Colony  ...        lat         lon
0     GUADALAJARA                 PARQUES DEL NILO  ...  20.654882 -103.279388
1     TLAQUEPAQUE                      SANTA ANITA  ...  20.552294 -103.438563
2     TLAQUEPAQUE  PARQUES DE SANTA CRUZ DEL VALLE  ...  20.566879 -103.350010
3     GUADALAJARA                 JARDINES EL SAUZ  ...  20.627611 -103.391443
4      TLAJOMULCO              FORESTA SANTA ANITA  ...  20.568879 -103.489603
...           ...                              ...  ...        ...         ...
1080  GUADALAJARA                    RANCHO BLANCO  ...  20.635726 -103.321554
1081      ZAPOPAN      LOS FLAMINGOS (LA VINATERA)  ...  20.838288 -103.441869
1082      ZAPOPAN                       EL SECRETO  ...  20.752440 -103.447605
1083  GUADALAJARA              HIGUERILLAS 1A SECC  ...  20.628798 -103.342275
1084      ZAPOPAN                        LA HUERTA  ...  20.739468 -103.380905

[1085 rows x 5 column

In [0]:
import io
def add_geometry_from_14JAL():
    cp_dict = {}
    val2 = []

    obj = s3_client.get_object(Bucket=bucket_name, Key='Data/Clean/14-Jal.geojson')
    data = json.loads(obj["Body"].read().decode("utf-8"))
    keys = jmespath.search("features[].properties.d_codigo", data)
    values = jmespath.search(str("features[].geometry"), data)

    for val in values:
        val2.append(str(val))

    cp_dict = dict(zip(keys, val2))
    df = pd.DataFrame.from_dict(cp_dict, orient="index", columns=["geometry"])
    df["CP"] = df.index
    df.reset_index(drop=True, inplace=True)

    cp_dict = pd.read_csv("./Data/Clean/cp_dict.csv")

    cp_dict["CP"] = cp_dict["CP"].astype("int32")

    cp_dict = cp_dict.merge(df[["CP", "geometry"]], on=["CP"], how="left")
    cp_dict.drop_duplicates(subset=["CP"], inplace=True)
    cp_dict.drop(cp_dict.index[cp_dict["CP"] == 44739], inplace=True)

    cp = gpd.read_file(bucket_uri+"/Data/Clean/14-Jal.geojson")  # one shape per CP

    # Compute in meters (UTM zone 13N covers Guadalajara), then convert back to lat/lon
    metric = cp.to_crs("EPSG:32613")

    cent = metric.centroid.to_crs("EPSG:4326")  # geometric center
    inside = metric.representative_point().to_crs("EPSG:4326")  # always inside the shape

    # Use the centroid unless it falls outside the shape; then use the inside point
    outside = ~metric.centroid.within(metric.geometry)
    pt = cent.where(~outside, inside)

    cp["lat"] = pt.y
    cp["lon"] = pt.x

    cp_df = cp[["d_codigo", "lat", "lon"]]
    cp_df.rename({"d_codigo": "CP"}, inplace=True, axis=1)

    cp_dict = pd.read_csv("./Data/Clean/cp_dict.csv")
    cp_dict = cp_dict.merge(cp_df[["CP", "lat", "lon"]], on=["CP"], how="left")
    cp_dict.dropna(inplace=True)
    cp_dict.to_csv("./Data/Clean/cp_dict.csv", index=False)
    
    print(cp_dict.head)


#add_geometry_from_14JAL()

def add_cp_coords_to_ALL():
    
    cp_dict = pd.read_csv("./Data/Clean/cp_dict.csv")

    df = pd.read_csv("./Data/Clean/ALL_Venta.csv")
    df = df.merge(
        cp_dict[["Municipality", "CP", "lat", "lon"]],
        on=["Municipality", "CP"],
        how="left",
    )

    df.dropna(inplace=True)
    df.to_csv(file, index=False)

    

#add_cp_coords_to_ALL()

Outlier removal with robust (MAD) z-scores: price per m² by Type+CP(+Year), then terrain size by CP.

In [0]:
Z_THRESHOLD = 3.5  # common cutoff for robust (MAD) z-scores
MIN_GROUP = 15  # smallest group allowed to have its own statistics
MIN_MAD = 0.10  # minimum spread (log units, ~10%) so near-identical groups don't explode


def robust_stats_terrain(frame, cols):
    g = frame.groupby(cols)["log_terrain"]
    med = g.transform("median")
    mad = g.transform(lambda s: (s - s.median()).abs().median())
    n = g.transform("count")
    return med, mad, n


def robust_stats(frame, cols):
    g = frame.groupby(cols)["log_ppm2"]
    med = g.transform("median")
    mad = g.transform(lambda s: (s - s.median()).abs().median())
    n = g.transform("count")
    return med, mad, n


def remove_price_terrain_outliers():
    file = "./Data/Clean/ALL_Venta.csv"
    df = pd.read_csv(file)

    # Adding columns
    df["log_ppm2"] = np.log(df["ppm2"])
    df["robust_z"] = np.nan
    df["group_level"] = None

    # Necessary masks
    valid = df["Price"].gt(0) & df["Terrain"].gt(0)
    df["invalid_ratio"] = ~valid
    pending = valid.copy()

    GROUP_LEVELS = [  # most specific level first, broadest last
        ["Type", "CP", "Year"],  # same postal code, same year
        ["Type", "CP"],  # same postal code, any year
    ]

    for cols in GROUP_LEVELS:
        med, mad, n = robust_stats(df, cols)
        ok = pending[valid] & (n >= MIN_GROUP)
        idx = ok[ok].index
        # Minimum spread: when most listings in a group share one price, MAD is
        # ~0 and tiny differences would look extreme (0.10 in log units = ±10%)
        mad = mad.clip(lower=MIN_MAD)
        # 0.6745 rescales MAD so the score is comparable to a normal z-score
        df.loc[idx, "robust_z"] = 0.6745 * (df.loc[idx, "log_ppm2"] - med[idx]) / mad[idx]
        df.loc[idx, "group_level"] = "+".join(cols)
        pending.loc[idx] = False

    df["outlier"] = df["robust_z"].abs() > Z_THRESHOLD

    df = df[~df["outlier"]]
    df.drop(
        columns=["invalid_ratio", "group_level", "robust_z", "outlier", "log_ppm2"],
        inplace=True,
    )
    df.to_csv(file, index=False)


def remove_terrain_outliers():
    file = "./Data/Clean/ALL_Venta.csv"
    df = pd.read_csv(file)
    print(df.shape)
    # Adding columns
    df["log_terrain"] = np.log(df["Terrain"])
    df["robust_z"] = np.nan
    df["group_level"] = None

    # Necessary masks
    valid = df["Terrain"].gt(0)
    df["invalid_ratio"] = ~valid
    pending = valid.copy()

    GROUP_LEVELS = [  # most specific level first, broadest last
        ["CP"]  # same postal code, any year
    ]

    for cols in GROUP_LEVELS:
        med, mad, n = robust_stats_terrain(df, cols)
        ok = pending[valid] & (n >= MIN_GROUP)
        idx = ok[ok].index
        mad = mad.clip(lower=MIN_MAD)
        df.loc[idx, "robust_z"] = (
            0.6745 * (df.loc[idx, "log_terrain"] - med[idx]) / mad[idx]
        )
        df.loc[idx, "group_level"] = "+".join(cols)
        pending.loc[idx] = False

    df["outlier"] = df["robust_z"].abs() > Z_THRESHOLD

    df = df[~df["outlier"]]
    df.drop(
        columns=["invalid_ratio", "group_level", "robust_z", "outlier", "log_terrain"],
        inplace=True,
    )
    df.to_csv(file, index=False)

    print(df.shape)



(1084024, 11)


(1042307, 11)


Train-station feature (from `transform.py`): for each CP, find the nearest train station (haversine distance) and turn the distance into an RBF similarity `closest_gamma`. Needs `./Data/Clean/train_stations.csv`.

In [0]:
def dist(lat1, long1, lat2, long2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians
    lat1, long1, lat2, long2 = map(radians, [lat1, long1, lat2, long2])
    # haversine formula
    dlon = long2 - long1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * asin(sqrt(a))
    # Radius of earth in kilometers is 6371
    km = 6371 * c
    return km


def find_nearest(stations_df, lat, long):

    distances = stations_df.apply(
        lambda row: dist(lat, long, row["lat"], row["lon"]), axis=1
    )

    return stations_df.loc[distances.idxmin(), ["lat", "lon"]]


def get_nearest_station():
    stations_df = pd.read_csv(bucket_uri+"/Data/Process/train_stations.csv")
    stations_df.drop(
        stations_df.index[stations_df["line"] == "Mi Macro Calzada"], inplace=True
    )

    stations_df.drop_duplicates("station", inplace=True)
    print(stations_df.head())
    cp_df = pd.read_csv("./Data/Clean/cp_dict.csv")

    cp_df.drop(
        columns=["closest_lat", "closest_lon"],
        errors="ignore",
        inplace=True,
    )

    closest_cords = cp_df.apply(
        lambda row: find_nearest(stations_df, row["lat"], row["lon"]),
        axis=1,
        result_type="expand",
    )

    closest_cords.rename(
        columns={"lon": "closest_lon", "lat": "closest_lat"}, inplace=True
    )

    cp_df = pd.concat([cp_df, closest_cords], axis="columns")

    cp_df.to_csv("./Data/Clean/cp_dict.csv", index=False)


def kfb_kernel_trainlines():
    cp_df = pd.read_csv("./Data/Clean/cp_dict.csv")

    cp_df.drop(
        columns=["closest_gamma"],
        errors="ignore",
        inplace=True,
    )

    coords_df = (
        cp_df[["lat", "lon", "closest_lat", "closest_lon"]]
        .dropna(subset=["lat"])
        .drop_duplicates(subset=["lat", "lon"])
        .copy()
    )

    gamma = 6000
    d2 = (coords_df["lat"] - coords_df["closest_lat"]) ** 2 + (
        coords_df["lon"] - coords_df["closest_lon"]
    ) ** 2

    coords_df["closest_gamma"] = np.exp(-gamma * d2)
    cp_df = cp_df.merge(
        coords_df, on=["lat", "lon", "closest_lat", "closest_lon"], how="left"
    )
    cp_df.to_csv("./Data/Clean/cp_dict.csv", index=False)



### Run the cleaning pipeline
Comment out any step you have already run. `source_replacement_map.csv` must exist in `./Data/Process/helpers/` before `clean_cp_dict()`.

In [0]:
create_frames_year_all()
add_shf_index()

make_cp_dict_unclean()
check_typos_cp_dict()
clean_sepomex_source()

clean_cp_dict()
clean_ALL_colonies()

add_geometry_from_14JAL()
add_cp_coords_to_ALL()

remove_price_terrain_outliers()
remove_terrain_outliers()

get_nearest_station()
kfb_kernel_trainlines()

ValueError: No objects to concatenate

Upload the clean data to S3.

In [0]:
for local_file in ["./Data/Clean/ALL_Venta.csv", "./Data/Clean/cp_dict.csv"]:
    uri = upload_to_s3(local_file, f"Data/Clean/{os.path.basename(local_file)}")
    print("Uploaded:", uri)

Uploaded: s3://sagemaker-us-east-2-205952664736/Data/Clean/ALL_Venta.csv
Uploaded: s3://sagemaker-us-east-2-205952664736/Data/Clean/cp_dict.csv


## Part 2 – Explore the data

In [0]:
venta_df = pd.read_csv("./Data/Clean/ALL_Venta.csv")

sample = venta_df.sample(n=2000, random_state=42).copy()
sample["log_price"] = np.log(sample["Price"])
sample["log_terrain"] = np.log(sample["Terrain"])
sample["log_ppm2"] = np.log(sample["ppm2"])

sns.pairplot(
    sample,
    vars=["log_price", "log_terrain", "log_ppm2", "shf_index"],
    hue="Type",
)

plt.show()

correlation = venta_df.corr(numeric_only=True)
# Create a correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(
    correlation,
    center=0,
    square=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
)
plt.show()

# Where the listings are: median log price per m2 by location
plt.figure(figsize=(10, 9))
loc = venta_df.groupby(["lat", "lon"])["ppm2"].median().reset_index()
plt.scatter(loc["lon"], loc["lat"], c=np.log(loc["ppm2"]), cmap="viridis", s=12)
plt.colorbar(label="log median price per m²")
plt.xlabel("lon")
plt.ylabel("lat")
plt.show()

In [0]:
df = venta_df
df.head(50)

## Part 3 – Features, train/validation/test split, save locally and to S3
Same features as `transform.py`: `closest_gamma` from `cp_dict` by CP, log of Terrain and Price, one-hot Type and Municipality, and drop `CP` and `ppm2`. The KMeans location-similarity columns are added inside the model (next sections), so the files and the endpoint only need these base columns.

The target column is `Price` (log of the price). Test is 20% (as in `transform.py`); validation is 20% of the rest, for the SageMaker tuner.

In [0]:
TARGET = "Price"

def group_lonely_cps(cp_col,cp_dict):

    count = cp_col.value_counts(ascending=True)
    count = count.where(count == 1).dropna()

    replacement_map = dict()


    for item, index in count.items():

        for shift in range(1, 20):
            new_cp = item + shift

            if new_cp not in cp_dict["CP"].values or new_cp in count.values:

                new_cp = item - shift

                if new_cp not in cp_dict["CP"].values or new_cp in count.values:
                    continue

            replacement_map.update({item: new_cp})
            break


    cp_col = cp_col.replace(replacement_map)

    return cp_col


def prepare_features(df, cp_dict):
    # cp_dict has one row per colony and several colonies share a CP;
    # keep one row per CP so the merge doesn't duplicate listings
    cp_feats = cp_dict[["closest_gamma", "CP"]].drop_duplicates(subset="CP")

    df = df.merge(cp_feats, on="CP", how="left")
    df.drop(columns=["ppm2"], inplace=True)
    df.dropna(inplace=True)

    df["Terrain"] = np.log(df["Terrain"])
    df["Price"] = np.log(df["Price"])

    df = pd.get_dummies(df, columns=["Type", "Municipality"], dtype=float)
    df["CP"] = group_lonely_cps(df['CP'],cp_dict)
    # Target first, then features
    return df[[TARGET] + [c for c in df.columns if c != TARGET]].reset_index(drop=True)


cp_dict = pd.read_csv("./Data/Clean/cp_dict.csv")
df = prepare_features(pd.read_csv("./Data/Clean/ALL_Venta.csv"), cp_dict)

train, test = train_test_split(df, test_size=0.20, random_state=42, stratify=df['CP'])
train_val, validation = train_test_split(train, test_size=0.20, random_state=42,stratify=train['CP'])

train.drop(columns=["CP"], inplace=True)
test.drop(columns=["CP"], inplace=True)
validation.drop(columns=["CP"], inplace=True)
train_val.drop(columns=["CP"], inplace=True)

os.makedirs("./Data/Model/", exist_ok=True)
train.to_csv("./Data/Model/venta_train.csv", index=False)
train_val.to_csv("./Data/Model/venta_trainval.csv", index=False)
validation.to_csv("./Data/Model/venta_validation.csv", index=False)
test.to_csv("./Data/Model/venta_test.csv", index=False)

# Upload the splits to S3 — keys must NOT start with "/" to avoid double-slash URIs
trainpath = upload_to_s3("./Data/Model/venta_train.csv", "Data/Clean/train.csv")
validationpath = upload_to_s3("./Data/Model/venta_validation.csv", "Data/Clean/validation.csv")
testpath = upload_to_s3("./Data/Model/venta_test.csv", "Data/Clean/test.csv")
trainvalpath = upload_to_s3("./Data/Model/venta_test.csv", "Data/Clean/venta_trainval.csv")

print("Features:", [c for c in df.columns if c != TARGET])
print(f"Train rows:      {len(train)}")
print(f"Validation rows: {len(validation)}")
print(f"Test rows:       {len(test)}")
print("trainpath:      ", trainpath)
print("validationpath: ", validationpath)
print("All splits uploaded to S3 successfully.")


Features: ['Terrain', 'Quarter', 'Year', 'shf_index', 'CP', 'lat', 'lon', 'closest_gamma', 'Type_Casa', 'Type_Departamento', 'Type_casa', 'Municipality_GUADALAJARA', 'Municipality_TLAJOMULCO', 'Municipality_TLAQUEPAQUE', 'Municipality_ZAPOPAN']
Train rows:      833835
Validation rows: 166767
Test rows:       208459
trainpath:       s3://sagemaker-us-east-2-205952664736/Data/Clean/train.csv
validationpath:  s3://sagemaker-us-east-2-205952664736/Data/Clean/validation.csv
All splits uploaded to S3 successfully.


## Part 4 – Train and test the Ridge model locally
Same model as `transform.py`: KMeans on lat/lon (`N_CLUSTERS = 280`), RBF similarity to each cluster center (`GAMMA_CL = 10000`), StandardScaler on X and y, Ridge with alpha from 10-fold GridSearchCV.

KMeans is fitted on the training rows only, so the test set stays unseen.

In [0]:
N_CLUSTERS = 280
GAMMA_CL = 10000
alphas = [10]


def add_cluster_similarity(x, km, gamma):
    sim_cols = [f"cl_sim_{i}" for i in range(km.n_clusters)]
    sims = pd.DataFrame(
        rbf_kernel(x[["lat", "lon"]], km.cluster_centers_, gamma=gamma),
        columns=sim_cols,
        index=x.index,
    )
    return pd.concat([x, sims], axis="columns")


train_df = pd.read_csv("./Data/Model/venta_trainval.csv")
test_df = pd.read_csv("./Data/Model/venta_test.csv")

x_train = train_df.drop(columns=TARGET)
y_train = train_df[[TARGET]]
x_test = test_df.drop(columns=TARGET)
y_test = test_df[[TARGET]]

km = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
km.fit(x_train[["lat", "lon"]])

x_train = add_cluster_similarity(x_train, km, GAMMA_CL)
x_test = add_cluster_similarity(x_test, km, GAMMA_CL)

std_scaler_x = StandardScaler()
std_scaler_y = StandardScaler()

x_train_s = std_scaler_x.fit_transform(x_train)
x_test_s = std_scaler_x.transform(x_test)

y_train_s = std_scaler_y.fit_transform(y_train).ravel()
y_test_s = std_scaler_y.transform(y_test).ravel()

folds = KFold(n_splits=10, shuffle=True, random_state=42)

ridge_grid = GridSearchCV(
    estimator=Ridge(),
    param_grid={"alpha": alphas},
    scoring="r2",
    cv=folds,
    n_jobs=-1,
)

ridge_grid.fit(x_train_s, y_train_s)

modelo_final = ridge_grid.best_estimator_

print("N_CLUSTERS:", N_CLUSTERS, "\tGAMMA:", GAMMA_CL)
print("Mejores hiperparámetros:", ridge_grid.best_params_)
print("R2 CV train:", ridge_grid.best_score_)

y_pred_train = std_scaler_y.inverse_transform(modelo_final.predict(x_train_s).reshape(-1, 1)).ravel()
y_pred_test = std_scaler_y.inverse_transform(modelo_final.predict(x_test_s).reshape(-1, 1)).ravel()

ytrain_real = y_train[TARGET].to_numpy()
ytest_real = y_test[TARGET].to_numpy()

metrics = {
    "N_CLUSTERS": N_CLUSTERS,
    "GAMMA": GAMMA_CL,
    "RIDGE_PARAMS": ridge_grid.best_params_,
    "R2_TRAIN": r2_score(ytrain_real, y_pred_train),
    "R2_TEST": r2_score(ytest_real, y_pred_test),
    "RMSE_TEST": np.sqrt(mean_squared_error(ytest_real, y_pred_test)),
    "MAE_TEST_PESOS": mean_absolute_error(np.exp(ytest_real), np.exp(y_pred_test)),
}

print(metrics)

# Save everything the model needs to predict in one file
model_path = "./model/"
os.makedirs(model_path, exist_ok=True)
joblib.dump(
    {
        "features": list(train_df.drop(columns=TARGET).columns),
        "kmeans": km,
        "gamma": GAMMA_CL,
        "scaler_x": std_scaler_x,
        "scaler_y": std_scaler_y,
        "ridge": modelo_final,
    },
    model_path + "model.joblib",
)
upload_to_s3(model_path + "model.joblib", f"Model/model.joblib")

KeyboardInterrupt: 

Create a script with all the steps and an argument parser so it can be used by the SageMaker SDK later.

In [0]:
%%writefile script.py

import argparse
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler


def add_cluster_similarity(x, km, gamma):
    sim_cols = [f"cl_sim_{i}" for i in range(km.n_clusters)]
    sims = pd.DataFrame(
        rbf_kernel(x[["lat", "lon"]], km.cluster_centers_, gamma=gamma),
        columns=sim_cols,
        index=x.index,
    )
    return pd.concat([x, sims], axis="columns")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    # Hyperparameters
    parser.add_argument("--alpha", type=float, default=10.0)
    parser.add_argument("--n-clusters", type=int, default=280)
    parser.add_argument("--gamma-cl", type=float, default=10000.0)
    # Data and model directories
    parser.add_argument(
        "--model-dir", type=str,
        default=os.environ.get("SM_MODEL_DIR", "./")
    )
    parser.add_argument(
        "--train", type=str,
        default=os.environ.get("SM_CHANNEL_TRAIN", "./")
    )
    parser.add_argument(
        "--validation", type=str,
        default=os.environ.get("SM_CHANNEL_VALIDATION", "./")
    )
    # Dataset filenames
    parser.add_argument("--train-file", type=str, default="train.csv")
    parser.add_argument("--validation-file", type=str, default="test.csv")
    parser.add_argument("--target", type=str, default="Price")
    args, _ = parser.parse_known_args()

    print("Loading datasets...")
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    val_df = pd.read_csv(os.path.join(args.validation, args.validation_file))

    # Separate features and target (target is log(Price))
    X_train = train_df.drop(columns=[args.target])
    y_train = train_df[[args.target]]
    X_val = val_df.drop(columns=[args.target])
    y_val = val_df[[args.target]]
    features = list(X_train.columns)
    print("Training:", X_train.shape)
    print("Validation:", X_val.shape)

    print("Location clusters...")
    km = KMeans(n_clusters=args.n_clusters, n_init=10, random_state=42)
    km.fit(X_train[["lat", "lon"]])
    X_train = add_cluster_similarity(X_train, km, args.gamma_cl)
    X_val = add_cluster_similarity(X_val, km, args.gamma_cl)

    print("Training model...")
    scaler_x = StandardScaler()
    scaler_y = StandardScaler()
    X_train_s = scaler_x.fit_transform(X_train)
    X_val_s = scaler_x.transform(X_val)
    y_train_s = scaler_y.fit_transform(y_train).ravel()

    ridge = Ridge(alpha=args.alpha)
    ridge.fit(X_train_s, y_train_s)

    # Evaluate model performance
    print("Evaluating model...")
    train_preds = scaler_y.inverse_transform(ridge.predict(X_train_s).reshape(-1, 1)).ravel()
    val_preds = scaler_y.inverse_transform(ridge.predict(X_val_s).reshape(-1, 1)).ravel()
    y_train = y_train[args.target].to_numpy()
    y_val = y_val[args.target].to_numpy()

    print(f"Train R2: {r2_score(y_train, train_preds):.4f}")
    print(f"Validation R2: {r2_score(y_val, val_preds):.4f}")
    print(f"Validation RMSE log: {np.sqrt(mean_squared_error(y_val, val_preds)):.4f}")
    print(f"Validation MAE pesos: {mean_absolute_error(np.exp(y_val), np.exp(val_preds)):.2f}")

    print("Saving model...")
    os.makedirs(args.model_dir, exist_ok=True)
    model_path = os.path.join(args.model_dir, "model.joblib")

    joblib.dump(
        {
            "features": features,
            "kmeans": km,
            "gamma": args.gamma_cl,
            "scaler_x": scaler_x,
            "scaler_y": scaler_y,
            "ridge": ridge,
        },
        model_path,
    )
    print(f"Model saved at: {model_path}")

Overwriting script.py


Necessary file for endpoint deployment. The endpoint receives the base feature columns as CSV (same order as the training files, without `Price`), adds the cluster similarities and returns the predicted price in pesos.

In [0]:
%%writefile inference.py
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import rbf_kernel


def model_fn(model_dir):
    model_path = os.path.join(model_dir, "model.joblib")

    print(f"Loading model from: {model_path}")

    model = joblib.load(model_path)

    print("Model loaded successfully.")
    return model


def predict_fn(input_data, model):
    x = pd.DataFrame(np.asarray(input_data, dtype=float).reshape(-1, len(model["features"])),
                     columns=model["features"])

    km = model["kmeans"]
    sims = rbf_kernel(x[["lat", "lon"]], km.cluster_centers_, gamma=model["gamma"])
    sim_cols = [f"cl_sim_{i}" for i in range(km.n_clusters)]
    x_full = pd.concat([x, pd.DataFrame(sims, columns=sim_cols)], axis="columns")

    pred_scaled = model["ridge"].predict(model["scaler_x"].transform(x_full))
    log_price = model["scaler_y"].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()

    # The model predicts log(Price); return the price in pesos
    return np.exp(log_price)

Writing inference.py


## Part 5 – SageMaker
Create a model using the SageMaker SDK; all the model artifacts are saved and can be used by SageMaker.

In [0]:
trainpath = f"s3://{bucket_name}/Data/Clean/train.csv"
validationpath = f"s3://{bucket_name}/Data/Clean/test.csv"

FRAMEWORK_VERSION = "1.4-2"
sklearn_estimator = SKLearn(
    entry_point="script.py",
    role=role,
    instance_count=1,
    instance_type="ml.c5.2xlarge",
    framework_version=FRAMEWORK_VERSION,
    base_job_name="ridge-venta",
    sagemaker_session=sagemaker_session,
    output_path=f"s3://{bucket_name}/Data/Model/output",
    hyperparameters={
        "alpha": 10,
        "n-clusters": 280,
        "gamma-cl": 10000,
    },
)

In [0]:


sklearn_estimator.fit(
    {
        "train": trainpath,
        "validation": validationpath
    },
    wait=True
)


2026-11-22 19:50:55 Starting - Starting the training job.

.

.


2026-11-22 19:51:21 Starting - Preparing the instances for training.

.

.

... (5 outputs left out)

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:1474: ConvergenceWarning: Number of distinct clusters (277) found smaller than n_clusters (280). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
Training model...



2026-11-22 19:54:29 Uploading - Uploading generated training modelEvaluating model...
Train R2: 0.8080
Validation R2: 0.8076
Validation RMSE log: 0.4304
Validation MAE pesos: 1518211.40
Saving model...
Model saved at: /opt/ml/model/model.joblib
2026-11-22 19:54:23,551 sagemaker-training-toolkit INFO     Reporting training SUCCESS



2026-11-22 19:54:37 Completed - Training job completed


Training seconds: 173
Billable seconds: 173


Obtain the best hyperparameters for the model using HyperparameterTuner: Ridge alpha, number of location clusters and the RBF gamma.

In [0]:
hyperparameter_ranges = {
    "alpha": ContinuousParameter(1, 1000, scaling_type="Logarithmic"),
    "n-clusters": IntegerParameter(100, 400),
    "gamma-cl": ContinuousParameter(1000, 20000, scaling_type="Logarithmic"),
}

Optimizer = HyperparameterTuner(
    estimator=sklearn_estimator,
    hyperparameter_ranges=hyperparameter_ranges,
    base_tuning_job_name="ridge-tuner-V2",
    objective_type="Maximize",
    objective_metric_name="validation-r2",
    metric_definitions=[
        {
            "Name": "validation-r2",
            "Regex": r"Validation R2: ([-0-9.]+)"
        }
    ],
    max_jobs=10,
    max_parallel_jobs=5
)

In [0]:


Optimizer.fit(
    {
        "train": trainpath,
        "validation": validationpath
    },
    wait=True
)


.

.

.

.

.

.

... (71 outputs left out)

.

.

.

!

In [0]:
results = Optimizer.analytics().dataframe()
display(results.sort_values("FinalObjectiveValue", ascending=False).head())
# Retrieve the best estimator
best_estimator = Optimizer.best_estimator()
print("Best training job:",
      best_estimator.latest_training_job.name)

artifact_path = sagemaker_session.describe_training_job(
    job_name=best_estimator.latest_training_job.name)["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact:", artifact_path)

,alpha,gamma-cl,n-clusters,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
4,1.300677,15810.615017,301.0,ridge-tuner-V2-261122-1823-006-cf03a989,Completed,0.8077,2026-11-22 18:26:16+00:00,2026-11-22 18:28:50+00:00,154.0
3,4.433833,6851.756080,314.0,ridge-tuner-V2-261122-1823-007-96f84ec6,Completed,0.8076,2026-11-22 18:26:25+00:00,2026-11-22 18:29:17+00:00,172.0
1,624.018587,10538.369677,258.0,ridge-tuner-V2-261122-1823-009-ca720e9a,Completed,0.8075,2026-11-22 18:27:11+00:00,2026-11-22 18:29:45+00:00,154.0
5,621.379789,7497.321320,258.0,ridge-tuner-V2-261122-1823-005-bf5f4f3e,Completed,0.8074,2026-11-22 18:23:55+00:00,2026-11-22 18:26:59+00:00,184.0
8,4.481018,3944.667894,155.0,ridge-tuner-V2-261122-1823-002-87b609ab,Completed,0.8071,2026-11-22 18:23:50+00:00,2026-11-22 18:26:08+00:00,138.0



2026-11-22 18:28:53 Starting - Found matching resource for reuse
2026-11-22 18:28:53 Downloading - Downloading the training image
2026-11-22 18:28:53 Training - Training image download completed. Training in progress.
2026-11-22 18:28:53 Uploading - Uploading generated training model
2026-11-22 18:28:53 Completed - Resource retained for reuse


Best training job: ridge-tuner-V2-261122-1823-006-cf03a989
Model artifact: s3://sagemaker-us-east-2-205952664736/Data/Model/output/ridge-tuner-V2-261122-1823-006-cf03a989/output/model.tar.gz


Deploy the model to SageMaker to have an endpoint to send data to and receive predictions.

In [0]:
print(repr(artifact_path))                      # checks for hidden spaces or odd characters
print(sagemaker_session.boto_region_name)       # should be us-east-2
print(sagemaker_session.default_bucket())       # the bucket the repack will upload to
print(sagemaker_session.default_bucket_prefix)



's3://sagemaker-us-east-2-205952664736/Data/Model/output/ridge-tuner-V2-261122-1823-006-cf03a989/output/model.tar.gz'
us-east-2
sagemaker-us-east-2-205952664736



In [0]:
import logging

for name in ("botocore", "boto3", "urllib3", "s3transfer"):
    lg = logging.getLogger(name)
    for h in list(lg.handlers):
        lg.removeHandler(h)
    lg.setLevel(logging.WARNING)

In [0]:
# artifact_path comes from the tuner cell; to reuse an older job, paste its path:
artifact_path = "s3://sagemaker-us-east-2-205952664736/Data/Model/output/ridge-tuner-V2-261122-1823-006-cf03a989/output/model.tar.gz"

model = SKLearnModel(
    model_data=artifact_path,
    entry_point="./inference.py",
    role=role,
    framework_version="1.4-2",
    sagemaker_session=sagemaker_session,
    name = 'RRegressionPriceModel'
)

In [0]:


predictor = model.deploy(
    instance_type="ml.c5.2xlarge",
    initial_instance_count=1,
    serializer=CSVSerializer(),
    deserializer=JSONDeserializer(),
    endpoint_name = 'Price-SG-ENDPOINT'
)



-

-

-

-

-

!

In [0]:
endpoint_name = predictor.endpoint_name
response = sagemaker_session.describe_endpoint(
    endpoint_name=endpoint_name
)
print("Endpoint:", endpoint_name)
print("Status:", response["EndpointStatus"])

Endpoint: Price-SG-ENDPOINT
Status: InService


Test endpoint

In [0]:
test = pd.read_csv("./Data/Model/venta_test.csv")
TARGET = 'Price'
print(test.head())

samples = test.sample(n=4, random_state=42)
X_samples = samples.drop(columns=[TARGET])

# Request predictions (price in pesos)
preds = predictor.predict(X_samples.values)

print("Predicted prices:", np.round(np.asarray(preds).ravel(), 0))
print("Actual prices:   ", np.round(np.exp(samples[TARGET].to_numpy()), 0))

       Price   Terrain  ...  Municipality_TLAQUEPAQUE  Municipality_ZAPOPAN
0  15.623799  5.075174  ...                       0.0                   1.0
1  15.093663  5.176150  ...                       0.0                   0.0
2  14.805052  4.653960  ...                       0.0                   1.0
3  14.688894  5.521461  ...                       0.0                   0.0
4  14.675866  4.727388  ...                       0.0                   0.0

[5 rows x 15 columns]
Predicted prices: [ 7158668.  7569402.  3962751. 12443113.]
Actual prices:    [ 6695000.  9480000.  2390000. 11900000.]


Evaluate using whole test dataset

In [0]:
def evaluate_endpoint(frame, batch_size=500):
    x = frame.drop(TARGET, axis=1)
    y_log = frame[TARGET].to_numpy()

    all_predictions = []
    for start in range(0, len(x), batch_size):
        batch = x.iloc[start:(start + batch_size)]
        predictions = predictor.predict(batch.values)
        all_predictions.extend(predictions)

    y_pred = np.asarray(all_predictions, dtype=float).ravel()

    return {
        "R2_log": r2_score(y_log, np.log(y_pred)),
        "RMSE_log": np.sqrt(mean_squared_error(y_log, np.log(y_pred))),
        "MAE_pesos": mean_absolute_error(np.exp(y_log), y_pred),
    }


results = evaluate_endpoint(test)
print(results)

{'R2_log': 0.8076562256257165, 'RMSE_log': 0.43037917655185287, 'MAE_pesos': 1518058.282101225}


In [0]:
test = pd.read_csv("./Data/Model/venta_test.csv")
print(test.columns)

Index(['Price', 'Terrain', 'Quarter', 'Year', 'shf_index', 'lat', 'lon',
       'closest_gamma', 'Type_Casa', 'Type_Departamento', 'Type_casa',
       'Municipality_GUADALAJARA', 'Municipality_TLAJOMULCO',
       'Municipality_TLAQUEPAQUE', 'Municipality_ZAPOPAN'],
      dtype='object')


Evaluate using validation dataset

In [0]:
val = pd.read_csv("./Data/Model/venta_validation.csv")

print(evaluate_endpoint(val))

{'R2_log': 0.8083995075784183, 'RMSE_log': 0.4298836954598128, 'MAE_pesos': 1522648.0696883106}


In [0]:
# Delete the real-time endpoint
predictor.delete_endpoint()

print("Endpoint deletion requested.")

Endpoint deletion requested.
